# seirgnn2 — `confirm`

Branch `exp/seir-gnn-v2` at `37703f68a3fb0673f47220944caccf7f48258c5e`. Selection is on validation only (plan R6); test numbers are written but not compared until a config is frozen.

In [ ]:
# Shell work goes through subprocess, not IPython magics: `tools/check_notebooks.py`
# parses every committed notebook as Python and runs in CI, so `!` and `%` cells
# fail the build.
import os, subprocess, sys
from pathlib import Path


def sh(*cmd, **kw):
    print("$", " ".join(str(c) for c in cmd), flush=True)
    r = subprocess.run([str(c) for c in cmd], text=True, **kw)
    if r.returncode:
        raise SystemExit(f"failed ({r.returncode}): {cmd}")
    return r


REPO_URL = "https://github.com/rathishTharusha/dengue-forecasting-gnn.git"
# Pinned to the commit this kernel was generated from, so the result is traceable.
SHA = "37703f68a3fb0673f47220944caccf7f48258c5e"
if not Path("repo").exists():
    sh("git", "clone", "--quiet", "--branch", "exp/seir-gnn-v2", REPO_URL, "repo")
os.chdir("repo")
sh("git", "checkout", "--quiet", SHA)
sh("git", "log", "--oneline", "-1")

In [ ]:
# Two installs, and the order matters.
#
# torch_geometric is a hard requirement of the five architectures and Kaggle does
# not ship it, so it is installed WITH its dependencies (it is pure Python; no
# build). Leaving it out is what broke kernel version 1: --no-deps on
# torch-geometric-temporal also skipped torch_geometric, so every graph arm died
# in its worker while the LSTM arms -- which need no PyG -- ran fine, and the
# grid came back 63 rows instead of 144 with no obvious error.
#
# torch-geometric-temporal then goes in WITHOUT dependencies, because its
# declared torch-sparse / torch-scatter have no wheels here and would try a
# source build. It needs torch_sparse only for evolvegcno, which none of the five
# use; backbones.install_shim() supplies the symbol.
sh(sys.executable, "-m", "pip", "install", "--quiet", "torch-geometric")
sh(sys.executable, "-m", "pip", "install", "--quiet", "--no-deps", "torch-geometric-temporal")
import torch
print("torch", torch.__version__, "| cuda", torch.cuda.is_available())
import torch_geometric; print("pyg", torch_geometric.__version__)

## All six encoders build and run

In [ ]:
# Gate: every encoder must build and run before the grid starts. Kernel v1 had no
# gate, so a broken install looked like a short results file rather than a failure.
out = subprocess.run([sys.executable, "seirgnn2/backbones.py"],
                     capture_output=True, text=True)
print(out.stdout or out.stderr)
assert "FAIL" not in out.stdout, "an encoder failed to build -- fix before running the grid"

## Run the `confirm` grid

In [ ]:
# Workers, not GPU: every architecture here is small and the grid is many short
# runs, so process-level parallelism over CPU cores beats one GPU stream. Set
# enable_gpu in kernel-metadata.json if a future grid actually needs it.
sh(sys.executable, "seirgnn2/sweep.py", "confirm", "--workers", "4", "--epochs", "400")

## Paired tests

In [ ]:
sh(sys.executable, "seirgnn2/stats.py", "confirm")

## Save

In [ ]:
import json, shutil
src = "seirgnn2/results/confirm.json"
shutil.copy(src, "/kaggle/working/confirm.json")
rows = json.load(open(src))
print(f"{len(rows)} rows -> /kaggle/working/confirm.json")